In [36]:
!pip install -U langchain langchain-community pypdf


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [37]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Sample_data/RAG.pdf")
docs = loader.load()

print(len(docs))


19


In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content[:500])

Number of chunks: 92
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁ


In [39]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
vector = embeddings.embed_query("What is Retrieval Augmented Generation?")

print(len(vector))

384


In [41]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created successfully")

Vector store created successfully


In [42]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [43]:
query = "What is Retrieval Augmented Generation?"

results = retriever.invoke(query)

print(results[0].page_content)

in 71% of cases, and a gold article is present in the top 10 retrieved articles in 90% of cases.
4.5 Additional Results
Generation Diversity Section 4.3 shows that RAG models are more factual and speciﬁc than
BART for Jeopardy question generation. Following recent work on diversity-promoting decoding
[33, 59, 39], we also investigate generation diversity by calculating the ratio of distinct ngrams to
total ngrams generated by different models. Table 5 shows that RAG-Sequence’s generations are
more diverse than RAG-Token’s, and both are signiﬁcantly more diverse than BART without needing
any diversity-promoting decoding.
Retrieval Ablations A key feature of RAG is learning to retrieve relevant information for the task.
To assess the effectiveness of the retrieval mechanism, we run ablations where we freeze the retriever
during training. As shown in Table 6, learned retrieval improves results for all tasks.


In [44]:
question = "What is RAG?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

print(context[:1000])

blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found
at https://huggingface.co/rag/
2https://github.com/pytorch/fairseq
3https://github.com/huggingface/transformers
17

blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found
at https://huggingface.co/rag/
2https://github.com/pytorch/fairseq
3https://github.com/huggingface/transformers
17

Broader Impact
This work offers several positive societal beneﬁts over previous work: the fact that it is more
strongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less
with generations that are more factual, and offers more control and interpretability. RAG could be
employed in a wide variety of scenarios with direct beneﬁt to society, for example by endowing it
with a medical index and asking it open-domain questions on that topic, or by helping people be more
effective at their jobs.
With these advantages also come potential downsides: Wikipedia,

In [45]:
chat_history = []

user_input = "What is RAG?"

answer = "RAG stands for Retrieval Augmented Generation."

chat_history.append(
    {"role":"user","content":user_input}
)

chat_history.append(
    {"role":"assistant","content":answer}
)

print(chat_history)

[{'role': 'user', 'content': 'What is RAG?'}, {'role': 'assistant', 'content': 'RAG stands for Retrieval Augmented Generation.'}]


In [46]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3",
    temperature=0
)

In [47]:
response = llm.invoke(
    "What is Retrieval Augmented Generation?"
)

print(response.content)

Retrieval-Augmented Generation (RAG) is a type of AI model that combines the strengths of both retrieval-based and generation-based approaches to generate text. In RAG, the model first retrieves relevant information from a large corpus or database, and then uses this retrieved information as input to generate new text.

The process can be broken down into two stages:

1. **Retrieval**: The model searches a large corpus or database for relevant information related to the input prompt or topic. This stage is similar to traditional retrieval-based models like search engines.
2. **Generation**: The retrieved information is then used as input to generate new text that expands, elaborates, or rephrases the original content. This stage is similar to generation-based models like language translation or text summarization.

RAG models have several advantages over traditional generation-only models:

1. **Improved accuracy**: By leveraging existing knowledge and information, RAG models can gener

In [48]:
def generate_answer(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    You are a helpful assistant.

    Use only the provided context to answer the question.

    If the answer is not present in the context, say:
    "I could not find the answer in the provided document."

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [49]:
answer = generate_answer(
    "What is Retrieval Augmented Generation?"
)

print(answer)

According to the provided context, Retrieval Augmented Generation (RAG) refers to hybrid generation models with access to parametric and non-parametric memory.


In [50]:
chat_history = []

In [51]:
chat_history = []

def chat(question):

    answer = generate_answer(question)

    chat_history.append({
        "role": "user",
        "content": question
    })

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [52]:
print(chat("What is RAG?"))

print(chat_history)

According to the provided context, RAG (Reinforced Analysis Generator) is a type of language model.
[{'role': 'user', 'content': 'What is RAG?'}, {'role': 'assistant', 'content': 'According to the provided context, RAG (Reinforced Analysis Generator) is a type of language model.'}]


In [53]:
def rewrite_question(chat_history, question):

    if not chat_history:
        return question

    history_text = "\n".join(
        [f"{msg['role']}: {msg['content']}"
         for msg in chat_history]
    )

    prompt = f"""
    Given the conversation history and the latest question,
    rewrite the latest question into a standalone question.

    Conversation:
    {history_text}

    Latest Question:
    {question}

    Standalone Question:
    """

    response = llm.invoke(prompt)

    return response.content.strip()

In [54]:
rewrite_question(
    [
        {"role":"user","content":"What is RAG?"},
        {"role":"assistant","content":"RAG is Retrieval Augmented Generation"}
    ],
    "Explain it more"
)

'What does "Retrieval Augmented Generation" (RAG) mean and how does it work?'

In [55]:
from tools import get_current_time
print(get_current_time())

2026-06-04 17:39:43


In [56]:
tool_schema = {
    "name": "get_current_time",
    "description": "Returns the current date and time",
    "parameters": {
        "type": "object",
        "properties": {}
    }
}

print(tool_schema)

{'name': 'get_current_time', 'description': 'Returns the current date and time', 'parameters': {'type': 'object', 'properties': {}}}


In [57]:
from tools import get_current_time

def execute_tool(tool_name):

    if tool_name == "get_current_time":
        return get_current_time()

    return "Unknown Tool"

In [58]:
print(execute_tool("get_current_time"))

2026-06-04 17:39:44


In [59]:
def rewrite_question(chat_history, question):

    if not chat_history:
        return question

    history_text = "\n".join(
        [f"{msg['role']}: {msg['content']}"
         for msg in chat_history]
    )

    prompt = f"""
    Given the conversation history and follow-up question,
    rewrite the follow-up question into a standalone question.

    Conversation:
    {history_text}

    Follow-up Question:
    {question}

    Standalone Question:
    """

    response = llm.invoke(prompt)

    return response.content.strip()

In [60]:
sample_history = [
    {
        "role":"user",
        "content":"What is Retrieval Augmented Generation?"
    },
    {
        "role":"assistant",
        "content":"RAG combines retrieval and generation."
    }
]

print(
    rewrite_question(
        sample_history,
        "Explain it more"
    )
)

What does RAG combine, exactly?


In [61]:
def route_question(question):

    if "time" in question.lower():
        return "tool"

    return "rag"

In [62]:
print(route_question("What is the current time?"))
print(route_question("What is RAG?"))

tool
rag


In [65]:
chat_history = []

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    standalone_question = rewrite_question(
        chat_history,
        user_input
    )

    route = route_question(
        standalone_question
    )

    if route == "tool":

        tool_name = "get_current_time"

        tool_result = execute_tool(tool_name)

        answer = f"Current time is {tool_result}"

    else:

        answer = generate_answer(
            standalone_question
        )

    chat_history.append(
        {
            "role":"user",
            "content":user_input
        }
    )

    chat_history.append(
        {
            "role":"assistant",
            "content":answer
        }
    )
    print("\nUser:", user_input)
    print("\nAssistant:", answer)


User: What is Retrieval Augmented Generation?

Assistant: According to the provided context, Retrieval Augmented Generation (RAG) refers to hybrid generation models with access to parametric and non-parametric memory.

User: Explain it more

Assistant: According to the provided context, having "access to parametric and non-parametric memory" in Retrieval Augmented Generation (RAG) models means that these models can utilize both types of memories.

Parametric memory refers to trainable parameters, which require far fewer trainable parameters for strong open-domain QA performance.

Non-parametric memory index does not consist of trainable parameters but consists of 21M 728-dimensional vectors, consisting of 15.3B values. These can be easily stored at 8-bit floating point precision to manage memory and disk footprints.

In other words, RAG models have access to both types of memories: one that is learned through training (parametric) and another that is based on a large index of pre-comp